# Projet Non-Supervisé : Analyse Cosmétique
## Classification automatique du type de peau vs expertise métier

**Objectif** : Remplacer l'expertise métier par un système de classement automatique basé sur des algorithmes de Machine Learning non-supervisés

## 1. Import des libraries et chargement des données

In [1]:
#%pip install --upgrade matplotlib scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# Configuration du style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Chargement des données
df = pd.read_csv('Projet_data_Cosmetique.csv', sep=';', decimal=',')
print("Dimensions du dataset:", df.shape)
print("\nPremières lignes:")
print(df.head())
print("\nTypes de données:")
print(df.dtypes)
print("\nValeurs manquantes:")
print(df.isnull().sum().sum())

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

## 2. Exploration et Préparation des données (EDA)

In [ ]:
# Analyse descriptive
print("Statistiques descriptives:")
print(df.describe())

# Distribution des types de peau
print("\nDistribution des types de peau (1=Grasse, 2=Normale, 3=Sèche):")
print(df['Peau'].value_counts().sort_index())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution par type de peau
df['Peau'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], color=['green', 'blue', 'orange'])
axes[0, 0].set_title('Distribution des types de peau', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Nombre de femmes')
axes[0, 0].set_xticklabels(['Grasse (1)', 'Normale (2)', 'Sèche (3)'], rotation=45)

# Distribution de l'âge
df['Age'].hist(ax=axes[0, 1], bins=15, color='skyblue', edgecolor='black')
axes[0, 1].set_title('Distribution de l\'âge', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Âge')

# Distribution IMC
df['IMC'].hist(ax=axes[1, 0], bins=15, color='lightcoral', edgecolor='black')
axes[1, 0].set_title('Distribution de l\'IMC', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('IMC')

# Box plot par type de peau
df.boxplot(column='Age', by='Peau', ax=axes[1, 1])
axes[1, 1].set_title('Âge par type de peau', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Type de peau')
axes[1, 1].set_ylabel('Âge')

plt.tight_layout()
plt.show()

# Variables principales par type de peau
print("\nMoyennes des variables principales par type de peau:")
print(df.groupby('Peau')[['Age', 'IMC', 'Co', 'Se', 'TE', 'pH']].mean())

## 3. Nettoyage et standardisation des données

In [ ]:
# Séparation de la variable cible et des features
y_true = df['Peau'].values  # Variable cible pour validation
X = df.drop(['Code', 'Peau'], axis=1)  # Toutes les features sauf Code et Peau

print(f"Nombre de features: {X.shape[1]}")
print(f"Nombre d'observations: {X.shape[0]}")

# Gestion des valeurs manquantes
print(f"\nValeurs manquantes avant imputation: {X.isnull().sum().sum()}")
X = X.fillna(X.mean())  # Imputation par la moyenne
print(f"Valeurs manquantes après imputation: {X.isnull().sum().sum()}")

# Standardisation (IMPORTANT pour PCA, K-means, DBSCAN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nMoyenne des variables standardisées: {X_scaled.mean(axis=0).round(10)[:5]}")
print(f"Écart-type des variables standardisées: {X_scaled.std(axis=0)[:5]}")

## 4. ANALYSE EN COMPOSANTES PRINCIPALES (ACP)

In [ ]:
# ACP complète pour explorer la variance
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

# Variance expliquée
variance_explained = pca_full.explained_variance_ratio_
cumsum_var = np.cumsum(variance_explained)

print("Variance expliquée par les 10 premiers PC:")
for i in range(min(10, len(variance_explained))):
    print(f"PC{i+1}: {variance_explained[i]:.4f} ({cumsum_var[i]:.4f} cumulée)")

# Détermination du nombre de composantes
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1
print(f"\nNombre de composantes pour 95% de variance: {n_components_95}")

# Visualisation de la variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].plot(range(1, len(variance_explained[:15])+1), variance_explained[:15], 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Composante Principale', fontsize=12)
axes[0].set_ylabel('Variance expliquée', fontsize=12)
axes[0].set_title('Scree Plot - Variance expliquée par PC', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, len(cumsum_var[:15])+1), cumsum_var[:15], 'ro-', linewidth=2, markersize=8)
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% variance')
axes[1].axhline(y=0.90, color='b', linestyle='--', label='90% variance')
axes[1].set_xlabel('Nombre de Composantes Principales', fontsize=12)
axes[1].set_ylabel('Variance cumulée expliquée', fontsize=12)
axes[1].set_title('Variance cumulée', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ACP réduite à 2 composantes pour visualisation
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f"Variance expliquée par PC1 et PC2: {pca_2d.explained_variance_ratio_.sum():.4f}")
print(f"PC1: {pca_2d.explained_variance_ratio_[0]:.4f}")
print(f"PC2: {pca_2d.explained_variance_ratio_[1]:.4f}")

# Visualisation avec les types de peau réels
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_peau = {1: 'green', 2: 'blue', 3: 'orange'}
labels_peau = {1: 'Grasse', 2: 'Normale', 3: 'Sèche'}

# Par type de peau réel
for peau_type in [1, 2, 3]:
    mask = y_true == peau_type
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], 
                   c=colors_peau[peau_type], label=labels_peau[peau_type], 
                   s=100, alpha=0.6, edgecolors='black')

axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
axes[0].set_title('ACP 2D - Types de peau réels (expert)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Biplot - contributions des variables
loadings = pca_2d.components_.T * np.sqrt(pca_2d.explained_variance_)
for i in range(len(X.columns)):
    axes[1].arrow(0, 0, loadings[i, 0]*3, loadings[i, 1]*3, 
                 head_width=0.1, head_length=0.1, fc='gray', ec='gray', alpha=0.5)
    if i < 15:  # Afficher seulement les premières variables
        axes[1].text(loadings[i, 0]*3.2, loadings[i, 1]*3.2, X.columns[i], 
                    fontsize=8, ha='center', va='center')

axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
axes[1].set_title('Biplot ACP - Contributions des variables', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

# Composantes principales principales
print("\nTop 5 variables contribuant à PC1:")
pc1_contrib = np.argsort(np.abs(pca_2d.components_[0]))[-5:]
for idx in pc1_contrib[::-1]:
    print(f"{X.columns[idx]}: {pca_2d.components_[0][idx]:.4f}")

print("\nTop 5 variables contribuant à PC2:")
pc2_contrib = np.argsort(np.abs(pca_2d.components_[1]))[-5:]
for idx in pc2_contrib[::-1]:
    print(f"{X.columns[idx]}: {pca_2d.components_[1][idx]:.4f}")

## 5. K-MEANS CLUSTERING

In [ ]:
# Détermination du nombre optimal de clusters (Méthode du coude)
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

print("K-Means Silhouette Scores:")
for k, score in zip(K_range, silhouette_scores):
    print(f"K={k}: {score:.4f}")

# Visualisation de la méthode du coude
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertie (Within-cluster sum of squares)', fontsize=12)
axes[0].set_title('Méthode du coude - K-Means', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Nombre de clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score - K-Means', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# K-Means optimal (3 clusters pour correspondre aux 3 types de peau)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

print(f"\nK-Means avec k={optimal_k}:")
print(f"Inertie: {kmeans.inertia_:.4f}")
print(f"Silhouette Score: {silhouette_score(X_scaled, kmeans_labels):.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, kmeans_labels):.4f}")
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X_scaled, kmeans_labels):.4f}")
print(f"\nDistribution des clusters K-Means:")
for i in range(optimal_k):
    print(f"Cluster {i}: {np.sum(kmeans_labels == i)} observations")

In [ ]:
# Visualisation K-Means
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Clusters K-Means
colors_clusters = ['red', 'purple', 'cyan']
for i in range(optimal_k):
    mask = kmeans_labels == i
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], 
                   c=colors_clusters[i], label=f'Cluster {i}', 
                   s=100, alpha=0.6, edgecolors='black')

# Centroïdes
centroid_pca = pca_2d.transform(kmeans.cluster_centers_)
axes[0].scatter(centroid_pca[:, 0], centroid_pca[:, 1], 
               c='yellow', marker='*', s=500, edgecolors='black', linewidth=2, 
               label='Centroïdes')

axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
axes[0].set_title(f'K-Means Clustering (k={optimal_k})', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matrice de confusion: clusters K-Means vs types de peau réels
cm = confusion_matrix(y_true, kmeans_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], 
            xticklabels=[f'Cluster {i}' for i in range(optimal_k)],
            yticklabels=['Grasse (1)', 'Normale (2)', 'Sèche (3)'])
axes[1].set_title('Confusion Matrix: Types réels vs Clusters K-Means', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Type de peau réel (expert)', fontsize=11)
axes[1].set_xlabel('Cluster K-Means', fontsize=11)

plt.tight_layout()
plt.show()

# Métriques de validation
ari_kmeans = adjusted_rand_score(y_true, kmeans_labels)
nmi_kmeans = normalized_mutual_info_score(y_true, kmeans_labels)
print(f"\nMétriques de comparaison K-Means vs expertise métier:")
print(f"Adjusted Rand Index: {ari_kmeans:.4f}")
print(f"Normalized Mutual Information: {nmi_kmeans:.4f}")

## 6. CLASSIFICATION ASCENDANTE HIÉRARCHIQUE (CAH)

In [ ]:
# Calcul de la matrice de distances
print("Calcul de la matrice de distances (cela peut prendre quelques secondes)...")
distances = pdist(X_scaled, metric='euclidean')

# Dendrogramme pour différentes méthodes de liaison
methods = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, method in enumerate(methods):
    Z = linkage(distances, method=method)
    dendrogram(Z, ax=axes[idx], no_labels=True, color_threshold=0)
    axes[idx].set_title(f'Dendrogramme - Méthode {method.upper()}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Indice d\'observation')
    axes[idx].set_ylabel('Distance')

plt.tight_layout()
plt.show()

print("Dendrogrammes générés pour les 4 méthodes de liaison")

In [ ]:
# CAH avec Ward (méthode optimale)
print("CAH avec méthode Ward...")
Z_ward = linkage(distances, method='ward')

# Détermination du nombre optimal de clusters
cah_metrics = []
for n_clusters in range(2, 11):
    cah = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    cah_labels = cah.fit_predict(X_scaled)
    sil_score = silhouette_score(X_scaled, cah_labels)
    cah_metrics.append({'n_clusters': n_clusters, 'silhouette': sil_score})
    print(f"n_clusters={n_clusters}: Silhouette={sil_score:.4f}")

# CAH optimal
optimal_n_cah = 3
cah = AgglomerativeClustering(n_clusters=optimal_n_cah, linkage='ward')
cah_labels = cah.fit_predict(X_scaled)

print(f"\nCAH avec {optimal_n_cah} clusters:")
print(f"Silhouette Score: {silhouette_score(X_scaled, cah_labels):.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, cah_labels):.4f}")
print(f"\nDistribution des clusters CAH:")
for i in range(optimal_n_cah):
    print(f"Cluster {i}: {np.sum(cah_labels == i)} observations")

In [ ]:
# Visualisation CAH
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Clusters CAH en espace PCA
colors_cah = ['darkgreen', 'darkblue', 'darkorange']
for i in range(optimal_n_cah):
    mask = cah_labels == i
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], 
                   c=colors_cah[i], label=f'Cluster {i}', 
                   s=100, alpha=0.6, edgecolors='black')

axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
axes[0].set_title(f'CAH Clustering (Ward, {optimal_n_cah} clusters)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matrice de confusion: clusters CAH vs types de peau réels
cm_cah = confusion_matrix(y_true, cah_labels)
sns.heatmap(cm_cah, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=[f'Cluster {i}' for i in range(optimal_n_cah)],
            yticklabels=['Grasse (1)', 'Normale (2)', 'Sèche (3)'])
axes[1].set_title('Confusion Matrix: Types réels vs Clusters CAH', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Type de peau réel (expert)', fontsize=11)
axes[1].set_xlabel('Cluster CAH', fontsize=11)

plt.tight_layout()
plt.show()

# Métriques de validation
ari_cah = adjusted_rand_score(y_true, cah_labels)
nmi_cah = normalized_mutual_info_score(y_true, cah_labels)
print(f"\nMétriques de comparaison CAH vs expertise métier:")
print(f"Adjusted Rand Index: {ari_cah:.4f}")
print(f"Normalized Mutual Information: {nmi_cah:.4f}")

## 7. DBSCAN CLUSTERING

In [ ]:
# Détermination des paramètres eps et min_samples
from sklearn.neighbors import NearestNeighbors

# K-distance graph
k = 4  # min_samples
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_scaled)
distances_knn, indices = neighbors_fit.kneighbors(X_scaled)
distances_knn = np.sort(distances_knn[:, k-1], axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(distances_knn)
ax.set_ylabel('4-NN Distance', fontsize=12)
ax.set_xlabel('Indice d\'observation', fontsize=12)
ax.set_title('K-distance Graph (k=4) pour déterminer eps', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Min distance: {distances_knn.min():.4f}")
print(f"Max distance: {distances_knn.max():.4f}")
print(f"Mean distance: {distances_knn.mean():.4f}")
print(f"Std distance: {distances_knn.std():.4f}")

In [ ]:
# Test de différentes valeurs d'eps
eps_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
min_samples = 4

dbscan_results = []

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, eps in enumerate(eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan_labels = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = list(dbscan_labels).count(-1)
    
    dbscan_results.append({
        'eps': eps,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'labels': dbscan_labels
    })
    
    print(f"eps={eps}: clusters={n_clusters}, bruit={n_noise}")
    
    # Visualisation
    colors = plt.cm.Spectral(np.linspace(0, 1, n_clusters + 1))
    for i in range(n_clusters):
        mask = dbscan_labels == i
        axes[idx].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], 
                         c=[colors[i]], label=f'Cluster {i}', s=100, alpha=0.6, edgecolors='black')
    
    # Bruit (label = -1)
    if n_noise > 0:
        mask_noise = dbscan_labels == -1
        axes[idx].scatter(X_pca_2d[mask_noise, 0], X_pca_2d[mask_noise, 1], 
                         c='red', marker='x', s=150, label='Bruit', linewidths=2)
    
    axes[idx].set_xlabel(f'PC1', fontsize=10)
    axes[idx].set_ylabel(f'PC2', fontsize=10)
    axes[idx].set_title(f'DBSCAN (eps={eps}, clusters={n_clusters}, bruit={n_noise})', fontsize=11, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN optimal (eps=2.0 semble bon)
optimal_eps = 2.0
optimal_min_samples = 4

dbscan_optimal = DBSCAN(eps=optimal_eps, min_samples=optimal_min_samples)
dbscan_labels = dbscan_optimal.fit_predict(X_scaled)

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_dbscan = list(dbscan_labels).count(-1)

print(f"\nDBSCAN optimal (eps={optimal_eps}, min_samples={optimal_min_samples}):")
print(f"Nombre de clusters: {n_clusters_dbscan}")
print(f"Nombre de points de bruit: {n_noise_dbscan}")
print(f"Distribution des clusters:")
for i in range(n_clusters_dbscan):
    print(f"Cluster {i}: {np.sum(dbscan_labels == i)} observations")

# Silhouette score (sans les points de bruit)
if n_clusters_dbscan >= 2 and n_noise_dbscan < len(X_scaled):
    mask_valid = dbscan_labels != -1
    if np.sum(mask_valid) > 0:
        sil_score_dbscan = silhouette_score(X_scaled[mask_valid], dbscan_labels[mask_valid])
        print(f"Silhouette Score (sans bruit): {sil_score_dbscan:.4f}")

In [ ]:
# Visualisation DBSCAN optimal
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Clusters DBSCAN
colors_dbscan = plt.cm.Spectral(np.linspace(0, 1, n_clusters_dbscan + 1))
for i in range(n_clusters_dbscan):
    mask = dbscan_labels == i
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], 
                   c=[colors_dbscan[i]], label=f'Cluster {i}', 
                   s=100, alpha=0.6, edgecolors='black')

# Bruit
if n_noise_dbscan > 0:
    mask_noise = dbscan_labels == -1
    axes[0].scatter(X_pca_2d[mask_noise, 0], X_pca_2d[mask_noise, 1], 
                   c='red', marker='X', s=200, label='Bruit', edgecolors='darkred', linewidths=2)

axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
axes[0].set_title(f'DBSCAN (eps={optimal_eps}, min_samples={optimal_min_samples})', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matrice de confusion (sans les points de bruit)
if n_noise_dbscan > 0:
    mask_valid = dbscan_labels != -1
    y_true_valid = y_true[mask_valid]
    dbscan_labels_valid = dbscan_labels[mask_valid]
    cm_dbscan = confusion_matrix(y_true_valid, dbscan_labels_valid)
else:
    cm_dbscan = confusion_matrix(y_true, dbscan_labels)

sns.heatmap(cm_dbscan, annot=True, fmt='d', cmap='Purples', ax=axes[1],
            xticklabels=[f'Cluster {i}' for i in range(n_clusters_dbscan)],
            yticklabels=['Grasse (1)', 'Normale (2)', 'Sèche (3)'])
axes[1].set_title('Confusion Matrix: Types réels vs Clusters DBSCAN', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Type de peau réel (expert)', fontsize=11)
axes[1].set_xlabel('Cluster DBSCAN', fontsize=11)

plt.tight_layout()
plt.show()

# Métriques (si pas trop de bruit)
if n_noise_dbscan < len(X_scaled) * 0.5:  # Si moins de 50% de bruit
    mask_valid = dbscan_labels != -1
    ari_dbscan = adjusted_rand_score(y_true[mask_valid], dbscan_labels[mask_valid])
    nmi_dbscan = normalized_mutual_info_score(y_true[mask_valid], dbscan_labels[mask_valid])
    print(f"\nMétriques de comparaison DBSCAN vs expertise métier (sans bruit):")
    print(f"Adjusted Rand Index: {ari_dbscan:.4f}")
    print(f"Normalized Mutual Information: {nmi_dbscan:.4f}")

## 8. RÈGLES D'ASSOCIATION

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Préparation des données pour les règles d'association
X_discrete = pd.DataFrame(X_scaled, columns=X.columns)

# Discrétisation en quartiles
for col in X_discrete.columns:
    X_discrete[col] = pd.qcut(X_discrete[col], q=3, labels=['Low', 'Medium', 'High'], duplicates='drop')

# Ajouter le type de peau réel
skin_labels = {1: 'Peau_Grasse', 2: 'Peau_Normale', 3: 'Peau_Sèche'}
X_discrete['Peau'] = y_true
X_discrete['Peau'] = X_discrete['Peau'].map(skin_labels)

# Ajouter les clusters
X_discrete['Cluster_KMeans'] = 'KM_Cluster_' + kmeans_labels.astype(str)
X_discrete['Cluster_CAH'] = 'CAH_Cluster_' + cah_labels.astype(str)
X_discrete['Cluster_DBSCAN'] = ['DBSCAN_Cluster_' + str(l) if l != -1 else 'DBSCAN_Bruit' for l in dbscan_labels]

print("Données discrétisées:")
print(X_discrete.head(10))
print(f"\nForme: {X_discrete.shape}")

In [ ]:
# Sélectionner quelques variables clés pour les règles d'association
key_columns = ['Age', 'IMC', 'Co', 'Se', 'TE', 'pH', 'Peau', 'Cluster_KMeans', 'Cluster_CAH']
X_assoc = X_discrete[key_columns]

# Convertir en format one-hot
X_encoded = pd.get_dummies(X_assoc, prefix=key_columns)

print(f"Variables encodées: {X_encoded.shape[1]}")
print(X_encoded.head())

# Apriori
print("\nCalcul des itemsets fréquents (apriori)...")
frequent_itemsets = apriori(X_encoded, min_support=0.1, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f"\nNombre d'itemsets fréquents: {len(frequent_itemsets)}")
print("\nTop 15 itemsets fréquents:")
print(frequent_itemsets.head(15))

In [ ]:
# Règles d'association
if len(frequent_itemsets) > 0:
    print("Calcul des règles d'association...")
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
    
    if len(rules) > 0:
        rules['antecedent_len'] = rules['antecedents'].apply(lambda x: len(x))
        rules['consequent_len'] = rules['consequents'].apply(lambda x: len(x))
        rules = rules.sort_values('lift', ascending=False)
        
        print(f"\nNombre de règles: {len(rules)}")
        print("Top 20 règles par Lift:")
        
        for idx, row in rules.head(20).iterrows():
            ant = ', '.join(list(row['antecedents']))
            cons = ', '.join(list(row['consequents']))
            print(f"\n{ant} => {cons}")
            print(f"  Support: {row['support']:.4f}, Confidence: {row['confidence']:.4f}, Lift: {row['lift']:.4f}")
    else:
        print("Aucune règle trouvée avec confidence >= 0.5")
else:
    print("Aucun itemset fréquent trouvé")

In [ ]:
# Visualisation des règles
if len(rules) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Support vs Confidence
    scatter = axes[0].scatter(rules['support'], rules['confidence'], 
                             c=rules['lift'], s=100, cmap='viridis', alpha=0.6, edgecolors='black')
    axes[0].set_xlabel('Support', fontsize=12)
    axes[0].set_ylabel('Confidence', fontsize=12)
    axes[0].set_title('Support vs Confidence (couleur = Lift)', fontsize=13, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=axes[0])
    cbar.set_label('Lift', fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Confidence vs Lift
    scatter2 = axes[1].scatter(rules['confidence'], rules['lift'], 
                              c=rules['support'], s=100, cmap='plasma', alpha=0.6, edgecolors='black')
    axes[1].set_xlabel('Confidence', fontsize=12)
    axes[1].set_ylabel('Lift', fontsize=12)
    axes[1].set_title('Confidence vs Lift (couleur = Support)', fontsize=13, fontweight='bold')
    cbar2 = plt.colorbar(scatter2, ax=axes[1])
    cbar2.set_label('Support', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 9. COMPARAISON DES ALGORITHMES

In [ ]:
# Tableau comparatif des algorithmes
comparison_data = {
    'Algorithme': ['K-Means', 'CAH (Ward)', 'DBSCAN'],
    'Clusters': [optimal_k, optimal_n_cah, n_clusters_dbscan],
    'Silhouette': [
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, cah_labels),
        silhouette_score(X_scaled[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1]) if np.sum(dbscan_labels != -1) > 0 else np.nan
    ],
    'Davies-Bouldin': [
        davies_bouldin_score(X_scaled, kmeans_labels),
        davies_bouldin_score(X_scaled, cah_labels),
        davies_bouldin_score(X_scaled[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1]) if np.sum(dbscan_labels != -1) > 0 else np.nan
    ],
    'Calinski-Harabasz': [
        calinski_harabasz_score(X_scaled, kmeans_labels),
        calinski_harabasz_score(X_scaled, cah_labels),
        calinski_harabasz_score(X_scaled[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1]) if np.sum(dbscan_labels != -1) > 0 else np.nan
    ],
    'ARI': [
        adjusted_rand_score(y_true, kmeans_labels),
        adjusted_rand_score(y_true, cah_labels),
        adjusted_rand_score(y_true[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1]) if np.sum(dbscan_labels != -1) > 0 else np.nan
    ],
    'NMI': [
        normalized_mutual_info_score(y_true, kmeans_labels),
        normalized_mutual_info_score(y_true, cah_labels),
        normalized_mutual_info_score(y_true[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1]) if np.sum(dbscan_labels != -1) > 0 else np.nan
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("COMPARAISON DES ALGORITHMES DE CLUSTERING")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Visualisation
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics = ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz', 'ARI', 'NMI']
algorithms = ['K-Means', 'CAH (Ward)', 'DBSCAN']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    values = comparison_df[metric].values
    colors_bars = ['#1f77b4', '#ff7f0e', '#2ca02c']
    bars = ax.bar(algorithms, values, color=colors_bars, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_ylabel(metric, fontsize=11, fontweight='bold')
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Ajouter les valeurs sur les barres
    for bar, value in zip(bars, values):
        if not np.isnan(value):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.3f}', ha='center', va='bottom', fontsize=10)

# Masquer le 6ème subplot vide
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 10. ANALYSE DES PROFILS DE PEAU

In [ ]:
# Analyse des profils par type de peau réel
X_with_skin = X.copy()
X_with_skin['Peau'] = y_true
X_with_skin['Peau_Type'] = X_with_skin['Peau'].map(labels_peau)

# Variables principales par type de peau
key_vars = ['Age', 'IMC', 'Co', 'Se', 'TE', 'pH', 'L_front', 'C_front']
profils_peau = X_with_skin.groupby('Peau_Type')[key_vars].mean()

print("\nProfils moyens par type de peau:")
print(profils_peau.round(3))

# Visualisation des profils
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.ravel()

for idx, var in enumerate(key_vars):
    data_by_skin = [X_with_skin[X_with_skin['Peau'] == i][var].values for i in [1, 2, 3]]
    bp = axes[idx].boxplot(data_by_skin, labels=['Grasse', 'Normale', 'Sèche'], patch_artist=True)
    
    for patch, color in zip(bp['boxes'], ['green', 'blue', 'orange']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    axes[idx].set_ylabel(var, fontsize=11, fontweight='bold')
    axes[idx].set_title(f'Distribution de {var}', fontsize=11)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 11. CONCLUSIONS ET RECOMMANDATIONS

In [ ]:
print("\n" + "="*80)
print("CONCLUSIONS ET RECOMMANDATIONS")
print("="*80)

print("\n1. RÉSUMÉ DES ANALYSES:")
print("-" * 80)
print(f"• Nombre total d'observations: {len(X)}")
print(f"• Nombre de variables: {X.shape[1]}")
print(f"• Variance expliquée par PC1+PC2: {pca_2d.explained_variance_ratio_.sum():.2%}")
print(f"• Variance expliquée par {n_components_95} composantes: 95%")

print("\n2. PERFORMANCE DES ALGORITHMES:")
print("-" * 80)
best_silhouette = comparison_df.loc[comparison_df['Silhouette'].idxmax()]
best_ari = comparison_df.loc[comparison_df['ARI'].idxmax()]
best_nmi = comparison_df.loc[comparison_df['NMI'].idxmax()]

print(f"\nMeilleur Silhouette: {best_silhouette['Algorithme']} ({best_silhouette['Silhouette']:.4f})")
print(f"Meilleur ARI: {best_ari['Algorithme']} ({best_ari['ARI']:.4f})")
print(f"Meilleur NMI: {best_nmi['Algorithme']} ({best_nmi['NMI']:.4f})")

print("\n3. CORRESPONDANCE AVEC L'EXPERTISE MÉTIER:")
print("-" * 80)
print(f"\nK-Means vs Expert (ARI={ari_kmeans:.4f}, NMI={nmi_kmeans:.4f}):")
print("  - ARI mesure la similarité ajustée (0=indépendant, 1=parfait)")
print("  - NMI mesure l'information mutuelle normalisée")

print(f"\nCAH (Ward) vs Expert (ARI={ari_cah:.4f}, NMI={nmi_cah:.4f}):")
print("  - Meilleure stabilité hiérarchique")
print("  - Permet de visualiser les dendrogrammes")

if n_noise_dbscan < len(X_scaled) * 0.5:
    print(f"\nDBSCAN vs Expert (ARI={ari_dbscan:.4f}, NMI={nmi_dbscan:.4f}):")
    print(f"  - {n_noise_dbscan} points identifiés comme bruit")
    print("  - Détection automatique du nombre de clusters")
else:
    print(f"\nDBSCAN: Trop de points de bruit ({n_noise_dbscan}) - non recommandé")

print("\n4. RECOMMANDATIONS POUR LA COMPAGNIE DE COSMÉTIQUES:")
print("-" * 80)
if ari_kmeans > 0.5:
    print("\n✓ K-Means peut remplacer l'expertise métier avec bon accord")
    print("  Avantages: Rapide, scalable, 3 clusters correspondent parfaitement")
else:
    print("\n✗ K-Means nécessite amélioration ou données supplémentaires")

if ari_cah > 0.5:
    print("\n✓ CAH offre une meilleure interprétabilité des résultats")
    print("  Avantages: Dendrogrammes explicatifs, résultats hiérarchiques")
else:
    print("\n✗ CAH montre des divergences avec l'expertise métier")

print("\n5. PROCHAINES ÉTAPES:")
print("-" * 80)
print("  1. Collecter plus de données pour améliorer la robustesse")
print("  2. Valider les résultats avec de nouvelles observations")
print("  3. Implémenter un système hybride (automatique + expertise)")
print("  4. Monitorer la performance en continu")
print("  5. Former les équipes à l'interprétation des résultats")
print("\n" + "="*80)